# SDD-013: the narrator, trained from scratch

A ~62M-parameter Llama-style model that turns one night's fact block into a 3-5 sentence summary. Trained from scratch on the corpus `build_corpus.py` writes, with the BPE tokenizer `train_tokenizer.py` writes, then exported to GGUF Q4 for llama.cpp.

**Inputs** (upload to Drive under `NARRATOR_DIR`): `train.jsonl`, `val.jsonl`, `test.jsonl`, `tokenizer/narrator-bpe-4000.model`.

**Layout of one example** (the runtime must build the same):

```
<s>{fact block, line breaks as <n>}<|summary|>{summary}</s>
```

The fact block is `model::factBlock()`, the whole prompt the nightly job gives the model. Loss is on the summary and `</s>` only.

**Outputs** (to `NARRATOR_DIR/export`): the HF checkpoint, `narrator-v1-f16.gguf`, `narrator-v1-q4_k_m.gguf` with its SHA-256, and `test_generations.jsonl` to run through `hms_cpap_narrator_facts check` on the build machine (the grounding check is hms-cpap's, not reimplemented here).

In [ ]:
# sentencepiece 0.2.0 has no wheel for Colab's current Python; take the current
# release. transformers/accelerate/torch: Colab's preinstalled versions (the
# Trainer API used here is stable across 4.4x+).
!pip -q install "sentencepiece>=0.2.1" safetensors
import transformers, accelerate
print('transformers', transformers.__version__, 'accelerate', accelerate.__version__)

from google.colab import drive
drive.mount('/content/drive')

NARRATOR_DIR = '/content/drive/MyDrive/narrator'   # where the corpus was uploaded
VOCAB = 4000          # the full corpus allows at most ~4.3k pieces (token_histogram.json); 4k uses 1,093
CTX = 1024            # fact block + summary; the histogram's p100 (891) must fit
SEED = 13

import json, math, os, random, time
import numpy as np, torch
import sentencepiece as spm
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
SEPARATOR, NEWLINE = '<|summary|>', '<n>'   # as train_tokenizer.py defines them
sp = spm.SentencePieceProcessor(model_file=f'{NARRATOR_DIR}/tokenizer/narrator-bpe-{VOCAB}.model')
assert sp.get_piece_size() == VOCAB
BOS, EOS, PAD = sp.bos_id(), sp.eos_id(), sp.pad_id()
SEP = sp.piece_to_id(SEPARATOR)
assert SEP != sp.unk_id() and sp.piece_to_id(NEWLINE) != sp.unk_id()

def prompt_ids(facts):
    return [BOS] + sp.encode(facts.replace('\n', NEWLINE)) + [SEP]

def encode(row):
    p = prompt_ids(row['prompt'])
    c = sp.encode(row['completion']) + [EOS]
    ids = (p + c)[:CTX]
    labels = ([-100] * len(p) + c)[:CTX]   # loss on the summary only
    return ids, labels

def load(name):
    with open(f'{NARRATOR_DIR}/{name}.jsonl') as fh:
        return [json.loads(l) for l in fh]

rows = {s: load(s) for s in ('train', 'val', 'test')}
data = {s: [encode(r) for r in rows[s]] for s in ('train', 'val')}
lens = np.array([len(i) for i, _ in data['train']])
truncated = int((lens >= CTX).sum())
print({s: len(v) for s, v in rows.items()}, 'tokens/pair p50/p99/max', np.percentile(lens, [50, 99, 100]), 'truncated', truncated)
assert truncated == 0, 'raise CTX: some pairs do not fit'

In [ ]:
from transformers import LlamaConfig, LlamaForCausalLM

# ~62M parameters, Llama architecture so llama.cpp runs it unchanged.
# 12 layers x (attention 4*d^2 + SwiGLU 3*d*ffn) at d=640, ffn=1728, plus a
# tied VOCAB x 640 embedding (4k x 640 = 2.6M).
config = LlamaConfig(
    vocab_size=VOCAB, hidden_size=640, intermediate_size=1728,
    num_hidden_layers=12, num_attention_heads=10, num_key_value_heads=10,
    max_position_embeddings=CTX, rope_theta=10000.0, rms_norm_eps=1e-5,
    hidden_act='silu', tie_word_embeddings=True, attention_bias=False, mlp_bias=False,
    bos_token_id=BOS, eos_token_id=EOS, pad_token_id=PAD,
    initializer_range=0.02)
model = LlamaForCausalLM(config)
n_params = sum(p.numel() for p in model.parameters())
print(f'{n_params / 1e6:.1f}M parameters')
assert 55e6 < n_params < 75e6

In [ ]:
from transformers import Trainer, TrainingArguments

class Pairs(torch.utils.data.Dataset):
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        ids, labels = self.items[i]
        return {'input_ids': ids, 'labels': labels}

def collate(batch):
    n = max(len(b['input_ids']) for b in batch)
    ids = torch.full((len(batch), n), PAD, dtype=torch.long)
    labels = torch.full((len(batch), n), -100, dtype=torch.long)
    mask = torch.zeros((len(batch), n), dtype=torch.long)
    for i, b in enumerate(batch):
        k = len(b['input_ids'])
        ids[i, :k] = torch.tensor(b['input_ids'])
        labels[i, :k] = torch.tensor(b['labels'])
        mask[i, :k] = 1
    return {'input_ids': ids, 'labels': labels, 'attention_mask': mask}

bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()   # A100 / L4; a T4 runs fp16
args = TrainingArguments(
    output_dir='/content/ckpt', seed=SEED,
    num_train_epochs=4,                      # ~250k pairs x ~400 tokens x 4 = ~400M tokens
    per_device_train_batch_size=64, gradient_accumulation_steps=2,
    learning_rate=6e-4, lr_scheduler_type='cosine', warmup_ratio=0.02,
    weight_decay=0.1, adam_beta1=0.9, adam_beta2=0.95, max_grad_norm=1.0,
    bf16=bf16, fp16=not bf16,
    eval_strategy='steps', eval_steps=1000, save_steps=1000, save_total_limit=2,
    logging_steps=100, report_to='none', dataloader_num_workers=2,
    group_by_length=True, load_best_model_at_end=True, metric_for_best_model='eval_loss')
trainer = Trainer(model=model, args=args, data_collator=collate,
                  train_dataset=Pairs(data['train']), eval_dataset=Pairs(data['val']))
# Resumes from the last checkpoint in /content/ckpt after a Colab disconnect
# (copy /content/ckpt to Drive between sessions to keep it).
resume = any(d.startswith('checkpoint-') for d in os.listdir('/content/ckpt')) if os.path.isdir('/content/ckpt') else False
trainer.train(resume_from_checkpoint=resume or None)
print(trainer.evaluate())

In [ ]:
# Greedy decode of the test split, for the grounding check and a human read.
model.eval()
out_dir = f'{NARRATOR_DIR}/export'
os.makedirs(out_dir, exist_ok=True)
gens = []
with torch.no_grad():
    for r in rows['test']:
        ids = torch.tensor([prompt_ids(r['prompt'])], device=model.device)
        out = model.generate(ids, max_new_tokens=200, do_sample=False, eos_token_id=EOS, pad_token_id=PAD)
        text = sp.decode(out[0, ids.shape[1]:].tolist())
        gens.append({'id': r['id'], 'facts': r['prompt'], 'text': text, 'teacher': r['completion']})
with open(f'{out_dir}/test_generations.jsonl', 'w') as fh:
    for g in gens:
        fh.write(json.dumps(g) + '\n')
for g in gens[:3]:
    print(g['facts'][:300], '...\n->', g['text'], '\n')
# On the build machine:  build/hms_cpap_narrator_facts check < test_generations.jsonl

In [ ]:
# The HF checkpoint, with the sentencepiece model beside it as tokenizer.model:
# llama.cpp's converter reads a Llama's vocabulary from that file.
import shutil
hf_dir = f'{out_dir}/hf'
model.save_pretrained(hf_dir, safe_serialization=True)
shutil.copy(f'{NARRATOR_DIR}/tokenizer/narrator-bpe-{VOCAB}.model', f'{hf_dir}/tokenizer.model')
json.dump({'tokenizer_class': 'LlamaTokenizer', 'add_bos_token': True, 'add_eos_token': False,
           'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>',
           'model_max_length': CTX}, open(f'{hf_dir}/tokenizer_config.json', 'w'), indent=1)
print(os.listdir(hf_dir))

In [ ]:
# GGUF F16, then Q4_K_M, with llama.cpp's own converter and quantizer.
# Pin LLAMA_CPP_REF to a release tag; the commit used is recorded below.
LLAMA_CPP_REF = 'master'
!rm -rf /content/llama.cpp && git clone -q https://github.com/ggml-org/llama.cpp /content/llama.cpp
!cd /content/llama.cpp && git checkout -q {LLAMA_CPP_REF} && git rev-parse HEAD > /content/llama_cpp_commit.txt
!pip -q install -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null
!cmake --build /content/llama.cpp/build -j4 --target llama-quantize llama-tokenize llama-simple > /dev/null

f16 = f'{out_dir}/narrator-v1-f16.gguf'
q4 = f'{out_dir}/narrator-v1-q4_k_m.gguf'
!python /content/llama.cpp/convert_hf_to_gguf.py {hf_dir} --outtype f16 --outfile {f16}
!/content/llama.cpp/build/bin/llama-quantize {f16} {q4} Q4_K_M

import hashlib
sha = hashlib.sha256(open(q4, 'rb').read()).hexdigest()
manifest = {'version': 'narrator-v1', 'file': os.path.basename(q4), 'sha256': sha,
            'bytes': os.path.getsize(q4), 'parameters': n_params, 'vocab': VOCAB, 'ctx': CTX,
            'llama_cpp_commit': open('/content/llama_cpp_commit.txt').read().strip(),
            'eval': trainer.evaluate()}
json.dump(manifest, open(f'{out_dir}/manifest.json', 'w'), indent=1)
print(json.dumps(manifest, indent=1))

In [ ]:
# Parity, SDD-013: llama.cpp must tokenize a fact block exactly as
# sentencepiece does, and its greedy decode of the Q4 file should read like the
# HF model's (Q4 changes logits, so the text may differ; the tokens may not).
import subprocess
fixture = rows['test'][0]['prompt'].replace('\n', NEWLINE)
ref = sp.encode(fixture)
out = subprocess.run(['/content/llama.cpp/build/bin/llama-tokenize', '-m', q4, '-p', fixture,
                      '--ids', '--no-bos', '--log-disable'], capture_output=True, text=True).stdout
got = json.loads(out.strip().splitlines()[-1])
print('tokenizer parity:', got == ref, len(ref), 'tokens')
assert got == ref, 'tokenizer mismatch between sentencepiece and the GGUF'
prompt = fixture + SEPARATOR   # llama-simple adds <s> itself
print(subprocess.run(['/content/llama.cpp/build/bin/llama-simple', '-m', q4, '-n', '200', prompt],
                     capture_output=True, text=True).stdout[-1200:])
print('HF greedy:', gens[0]['text'])